In [1]:
!pip install flask pyngrok --ignore-installed blinker

  Using cached werkzeug-3.0.4-py3-none-any.whl.metadata (3.7 kB)
  Using cached jinja2-3.1.4-py3-none-any.whl.metadata (2.6 kB)
  Using cached click-8.1.7-py3-none-any.whl.metadata (3.0 kB)
  Using cached PyYAML-6.0.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.1 kB)
  Using cached MarkupSafe-2.1.5-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 2.1 MB/s eta 0:00:00
Using cached click-8.1.7-py3-none-any.whl (97 kB)
Using cached jinja2-3.1.4-py3-none-any.whl (133 kB)
Using cached PyYAML-6.0.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (751 kB)
Using cached werkzeug-3.0.4-py3-none-any.whl (227 kB)
Using cached MarkupSafe-2.1.5-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (25 kB)
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
!ngrok authtoken 2n2WLvnbSoZ92IPApQk7W6eIVPh_D37YPsJ8CK8TdMNDJbx

In [2]:
# Required Imports
import os
import sys
import numpy as np
import tensorflow as tf
from keras.models import load_model
from keras.preprocessing.text import tokenizer_from_json
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import json
from flask import Flask, request, jsonify, render_template_string
from pyngrok import ngrok

In [3]:
# Initialize Flask App
app = Flask(__name__)

In [4]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive Mounted Successfully.")

sys.path.insert(0, '/content/drive/MyDrive/CSCK507_EMA_GroupC_Final')
import config

model_file = os.path.join(config.MAIN_DIR_PATH, config.SEQ2SEQ_WA_MODEL)
tokenizer_file = os.path.join(config.MAIN_DIR_PATH, config.TOKENIZER_FILE)

print(f"Model File Path: {model_file}")
print(f"Tokenizer File Path: {tokenizer_file}")

Mounted at /content/drive
Google Drive Mounted Successfully.
Model File Path: /content/drive/MyDrive/CSCK507_EMA_GroupC_Final/seq2seq_with_attention_model.h5
Tokenizer File Path: /content/drive/MyDrive/CSCK507_EMA_GroupC_Final/tokenizer.json


In [5]:
# Load the Pre-trained Model
seq2seq_model = load_model(model_file)
print("Model loaded successfully!")

# Print the model summary to verify its structure
seq2seq_model.summary()

Model loaded successfully!
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 Encoder_Input (InputLayer)  [(None, 30)]                 0         []                            
                                                                                                  
 Decoder_Input (InputLayer)  [(None, 30)]                 0         []                            
                                                                                                  
 encoder_embedding (Embeddi  (None, 30, 96)               6149049   ['Encoder_Input[0][0]']       
 ng)                                                      6                                       
                                                                                                  
 Decoder_Embedding (Embeddi  (None, 30, 96)               6149049  

In [6]:
# Load the tokenizer
with open(tokenizer_file, 'r') as f:
    tokenizer_data = json.load(f)
    tokenizer = tf.keras.preprocessing.text.tokenizer_from_json(tokenizer_data)

# Create index mappings using the correct word-level index
target_token_index = tokenizer.word_index
reverse_target_word_index = {i: word for word, i in target_token_index.items()}

print(f"Tokenizer Loaded. Number of Tokens: {len(target_token_index)}")

Tokenizer Loaded. Number of Tokens: 640525


In [7]:
# Extract the encoder
encoder_inputs = seq2seq_model.input[0]  # Encoder input is the first input
encoder_outputs, encoder_state = seq2seq_model.get_layer('Encoder_GRU').output

encoder_model = tf.keras.Model(encoder_inputs, [encoder_outputs, encoder_state])

# Extract the decoder
decoder_inputs = seq2seq_model.input[1]  # Decoder input is the second input
decoder_state_input_h = tf.keras.Input(shape=(64,), dtype='bfloat16', name='decoder_hidden_state')
encoder_outputs_input = tf.keras.Input(shape=(None, 64), dtype='bfloat16', name='encoder_outputs')

# Extract the embedding and GRU layers
decoder_embedding_layer = seq2seq_model.get_layer('Decoder_Embedding')
decoder_gru_layer = seq2seq_model.get_layer('Decoder_GRU')
attention_layer = seq2seq_model.get_layer('Attention_Concatenate')
dense_layer = seq2seq_model.get_layer('Decoder_Dense')

# Decoder embedding for the current input word
decoder_embedding_outputs = decoder_embedding_layer(decoder_inputs)

# Decoder GRU processing one time step at a time
decoder_gru_output, decoder_state_h = decoder_gru_layer(decoder_embedding_outputs, initial_state=[decoder_state_input_h])

# Compute the attention context vector
score = tf.matmul(decoder_gru_output, encoder_outputs_input, transpose_b=True)
attention_weights = tf.nn.softmax(score, axis=-1)
context_vector = tf.matmul(attention_weights, encoder_outputs_input)

# Concatenate the context vector with the decoder's GRU output
decoder_context_concat = tf.keras.layers.Concatenate(axis=-1)([decoder_gru_output, context_vector])

# Pass through the dense layer
decoder_outputs = dense_layer(decoder_context_concat)

# Define the decoder model for inference
decoder_model = tf.keras.Model(
    [decoder_inputs, decoder_state_input_h, encoder_outputs_input],
    [decoder_outputs, decoder_state_h, context_vector]
)

In [8]:
def batch_greedy_decode(input_seqs, encoder_model, decoder_model, target_token_index, reverse_target_word_index, max_decoder_seq_length):
    """
    Perform Greedy Search to decode a batch of sequences using the encoder and decoder models with attention.
    """
    batch_size = input_seqs.shape[0]

    # Encode the input sequences to get the encoder outputs and the initial state
    encoder_outputs, encoder_state = encoder_model.predict(input_seqs, verbose=0)

    # Initialize the target sequences with the <SOS> token
    start_token_index = target_token_index.get('<SOS>', 0)
    target_seqs = np.full((batch_size, 1), start_token_index, dtype='int32')

    decoded_sentences = [[] for _ in range(batch_size)]
    finished = [False] * batch_size
    decoder_state = encoder_state  # Initialize decoder state with encoder state

    for _ in range(max_decoder_seq_length):
        # Use the decoder to predict the next tokens, incorporating attention
        decoder_outputs, decoder_state, attention_context_vector = decoder_model.predict(
            [target_seqs, decoder_state, encoder_outputs],
            verbose=0
        )

        # Select the token with the highest probability
        sampled_token_indices = np.argmax(decoder_outputs[:, -1, :], axis=1)

        for i, sampled_token_index in enumerate(sampled_token_indices):
            if not finished[i]:
                sampled_word = reverse_target_word_index.get(sampled_token_index, '')

                if sampled_word == '<EOS>':
                    finished[i] = True
                else:
                    decoded_sentences[i].append(sampled_word)

        # Update the target sequences for the next iteration
        target_seqs = sampled_token_indices[:, np.newaxis]

    return [' '.join(sentence) for sentence in decoded_sentences]

def decode_indices_to_words(sequence, reverse_target_word_index):
    """
    Convert a sequence of indices to a readable string using the reverse target word index.
    """
    words = []
    for index in sequence:
        word = reverse_target_word_index.get(index, '')
        if word == '<EOS>':
            break
        words.append(word)

    return ' '.join(words).strip()

In [ ]:
# HTML Template for the Chatbot UI
html_template = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Chatbot UI</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            max-width: 600px;
            margin: auto;
        }
        #chat-box {
            border: 1px solid #ccc;
            height: 400px;
            overflow-y: scroll;
            padding: 10px;
            margin-bottom: 10px;
        }
        .user-message, .bot-message {
            margin: 10px 0;
            padding: 8px;
            border-radius: 8px;
        }
        .user-message {
            text-align: right;
            background-color: #e1f5fe;
        }
        .bot-message {
            text-align: left;
            background-color: #e8eaf6;
        }
    </style>
</head>
<body>
    <h2>Chat with the Bot</h2>
    <div id="chat-box"></div>
    <input type="text" id="user-input" placeholder="Type your message here..." style="width: 80%;">
    <button onclick="sendMessage()">Send</button>

    <script>
        function sendMessage() {
            const userInput = document.getElementById('user-input').value;
            if (!userInput) {
                return;
            }

            // Append the user's message to the chat-box
            const chatBox = document.getElementById('chat-box');
            const userMessageDiv = document.createElement('div');
            userMessageDiv.className = 'user-message';
            userMessageDiv.textContent = userInput;
            chatBox.appendChild(userMessageDiv);

            // Clear the input field
            document.getElementById('user-input').value = '';

            // Make the request to the backend
            fetch('/chat', {
                method: 'POST',
                headers: {
                    'Content-Type': 'application/json'
                },
                body: JSON.stringify({ input: userInput })
            })
            .then(response => response.json())
            .then(data => {
                // Append the bot's response to the chat-box
                const botMessageDiv = document.createElement('div');
                botMessageDiv.className = 'bot-message';
                botMessageDiv.textContent = data.response;
                chatBox.appendChild(botMessageDiv);

                // Scroll to the bottom of the chat-box
                chatBox.scrollTop = chatBox.scrollHeight;
            })
            .catch(error => {
                console.error('Error:', error);
            });
        }
    </script>
</body>
</html>
"""

# Flask Endpoint to Serve the Chatbot UI
@app.route('/')
def index():
    return render_template_string(html_template)

# Flask Endpoint to Handle User Input
@app.route('/chat', methods=['POST'])
def chat():
    user_input = request.json.get('input')
    if not user_input:
        return jsonify({'error': 'No input provided'}), 400

    input_tokens = tokenizer.texts_to_sequences([user_input])
    input_seq = tf.keras.preprocessing.sequence.pad_sequences(input_tokens, maxlen=30, padding='post')
    response = batch_greedy_decode(input_seq, encoder_model, decoder_model, target_token_index, reverse_target_word_index, max_decoder_seq_length=30)

    return jsonify({'response': response})

# Create a public URL using ngrok
public_url = ngrok.connect(5000)
print(f"Public URL: {public_url}")

# Run the Flask App
if __name__ == "__main__":
    app.run(port=5000)

Public URL: NgrokTunnel: "https://182a-35-226-207-48.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [07/Oct/2024 16:09:58] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Oct/2024 16:09:58] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [07/Oct/2024 16:12:03] "POST /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Oct/2024 16:13:13] "POST /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Oct/2024 16:14:02] "POST /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Oct/2024 16:14:51] "POST /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Oct/2024 16:15:35] "POST /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Oct/2024 16:25:16] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Oct/2024 17:24:14] "POST /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Oct/2024 17:25:09] "POST /chat HTTP/1.1" 200 -
INFO:werkz